# 03 Gold (A): Correlation analysis

**Questions**
1. Is delivery time linked to the review score?
2. Is a late delay (`delay_days`) linked to the review score?
3. Are item price and freight value related?

**Method:** Spearman correlation (rank-based, so it works with skewed data like price and delivery days, and with ordinal review scores).
**Data:** delivered orders only. Needs `gold.fact_orders`, `gold.fact_order_items` and `gold.fact_reviews`.

**How to read rho:** below 0.1 negligible, 0.1-0.3 weak, 0.3-0.5 moderate, above 0.5 strong. The sign shows the direction.

## 0. Setup

Import libraries and connect to the database. `db.py` lives in `notebooks/`, so adjust the `.parent` chain if this notebook sits at a different depth.

In [15]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text
from scipy.stats import spearmanr

# Walk up from the current folder until a folder containing db.py is found
p = Path.cwd()
while not (p / "db.py").exists():
    if p == p.parent:
        raise FileNotFoundError("db.py not found in any parent folder")
    p = p.parent
sys.path.append(str(p))

from db import get_engine
engine = get_engine()

Helper to run a SELECT and return a DataFrame.

In [16]:
def q(sql):
    """Run a SELECT and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Load the data

Order level: delivery columns joined with the review score. Only delivered orders have `delivery_days` and `delay_days`.

In [17]:
df = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT CONCAT(review_id, order_id)) AS unique_combinations
    FROM gold.fact_reviews
""", engine)

display(df)

,total_rows,unique_combinations
0,99224,99224


In [18]:
orders = q("""
SELECT f.order_id, f.delivery_days, f.delay_days, r.review_score_avg
FROM gold.fact_orders f
JOIN (
    -- One row per order: some orders have several reviews
    SELECT order_id, AVG(CAST(review_score AS FLOAT)) AS review_score_avg
    FROM gold.fact_reviews
    GROUP BY order_id
) r ON r.order_id = f.order_id
WHERE f.delivered_customer_date IS NOT NULL
""")
orders.shape

(95830, 4)

Item level: price and freight for items of delivered orders.

In [19]:
items = q("""
SELECT i.price, i.freight_value
FROM gold.fact_order_items i
JOIN gold.fact_orders f ON f.order_id = i.order_id
WHERE f.delivered_customer_date IS NOT NULL
""")
items.shape

(110196, 2)

## 2. Quick data check

Look at missing values and basic statistics before correlating. Many rows with NULLs would reduce the sample size.

In [20]:
orders.isna().sum()

order_id            0
delivery_days       0
delay_days          0
review_score_avg    0
dtype: int64

In [21]:
orders[["delivery_days", "delay_days", "review_score_avg"]].describe().round(2)

,delivery_days,delay_days,review_score_avg
count,95830.00,95830.00,95830.00
mean,12.46,-11.91,4.16
std,9.47,10.11,1.28
min,0.00,-147.00,1.00
25%,7.00,-17.00,4.00
50%,10.00,-12.00,5.00
75%,16.00,-7.00,5.00
max,208.00,188.00,5.00


In [22]:
items.describe().round(2)

,price,freight_value
count,110196.00,110196.00
mean,119.98,19.95
std,182.30,15.70
min,0.85,0.00
25%,39.90,13.08
50%,74.90,16.26
75%,134.17,21.15
max,6735.00,409.68


Drop rows with missing values, because Spearman needs complete pairs.

In [23]:
orders = orders.dropna()
items = items.dropna()
len(orders), len(items)

(95830, 110196)

## 3. Correlation function

One small function returns rho, p-value and a strength label. The p-value will be tiny with this many rows, so read rho and the label, not the p-value.

In [24]:
def corr(df, a, b, label):
    rho, p = spearmanr(df[a], df[b])
    strength = ("negligible" if abs(rho) < 0.1 else
                "weak"       if abs(rho) < 0.3 else
                "moderate"   if abs(rho) < 0.5 else "strong")
    return {"pair": label, "n": len(df), "rho": round(rho, 3),
            "p_value": f"{p:.2g}", "strength": strength}

## 4. Results

**Q1.** Delivery days vs review score. Expected: negative (longer delivery, lower score).

In [25]:
r1 = corr(orders, "delivery_days", "review_score_avg", "delivery days vs review score")
r1

{'pair': 'delivery days vs review score',
 'n': 95830,
 'rho': np.float64(-0.235),
 'p_value': '0',
 'strength': 'weak'}

**Q2.** Delay days (delivered date minus estimated date) vs review score. Customers judge against the promised date, so this is often stronger than Q1.

In [26]:
r2 = corr(orders, "delay_days", "review_score_avg", "delay days vs review score")
r2

{'pair': 'delay days vs review score',
 'n': 95830,
 'rho': np.float64(-0.176),
 'p_value': '0',
 'strength': 'weak'}

**Q3.** Price vs freight. Freight depends mostly on weight and distance, so a weak link is normal.

In [27]:
r3 = corr(items, "price", "freight_value", "price vs freight")
r3

{'pair': 'price vs freight',
 'n': 110196,
 'rho': np.float64(0.434),
 'p_value': '0',
 'strength': 'moderate'}

All three results in one table.

In [28]:
pd.DataFrame([r1, r2, r3])

,pair,n,rho,p_value,strength
0,delivery days vs review score,95830,-0.235,0,weak
1,delay days vs review score,95830,-0.176,0,weak
2,price vs freight,110196,0.434,0,moderate


## 5. Sanity check with averages

Spearman gives one number; averages make it concrete. Average review score by delay bucket (on time, 1-3, 4-7, 8+ days late).

In [29]:
bucket = pd.cut(
    orders["delay_days"],
    bins=[-1000, 0, 3, 7, 1000],
    labels=["on time", "1-3 days late", "4-7 days late", "8+ days late"]
)
orders.groupby(bucket, observed=True)["review_score_avg"].agg(["count", "mean"]).round(2)

,count,mean
delay_days,,
on time,89448,4.29
1-3 days late,1852,3.29
4-7 days late,1748,2.11
8+ days late,2782,1.70


## 6. Finding and decision

**Finding:** Delivery time and review score have a weak negative link (rho = -0.235 for delivery days, -0.176 for delay days, n = 95,830), but the buckets show why the overall number is low. Orders delivered on time average 4.29, while 1-3 days late drops to 3.29, 4-7 days late to 2.11, and 8+ days late to 1.70. The effect is a threshold: scores fall sharply once the promised date is missed, and among the 93% of orders that arrive on time, delivering earlier makes little difference. A single rank correlation over all orders understates this. Price and freight have a moderate positive link (rho = 0.434, n = 110,196): pricier items tend to cost more to ship, but the link is far from strong, since freight depends mostly on weight and distance.

**Decision:** The dashboard shows average review score by delay bucket (on time, 1-3, 4-7, 8+ days late), not a correlation or raw delivery days, because the relationship is a threshold effect. The states with the highest late rates from A2 (AL, MA, PI, CE) are where on-time improvements would raise scores most. Price and freight are not tied tightly enough to predict one from the other, so freight analysis should use weight or distance, not price.

**Caveats:** correlation is not cause (a late order may also be a larger or more complex order); delivered orders only; scores are order averages when an order has several reviews; p-values are 0 because of the large sample, so rho and the buckets carry the message.